# Management
I use a kaggle notebook for my IDE and to run. The code below creates .gitignore and other things I would normally do in my local IDE (pycharm), but I cannot do here.

In [7]:
%%writefile .gitignore
train_X/
train_y/
val_X/
val_y/

SyntaxError: invalid syntax (1491267869.py, line 3)

# CNN Encoder-Decoder
This code was originally purpoused for turning images took in the day to nightime images, but the dataset was unpaired and I then had to repurpouse. It now uses the edge to shoes dataset (Edges2Shoes on kaggle). This is a simple CNN that turns images to images using an encoder-decoder architecture. The training code is below.

In [ ]:
# Training code

import tensorflow as tf
import cv2
import os
from tqdm import tqdm
import shutil

def split_imgs(input_folder, output_A, output_B):
    if os.path.exists(output_A):
        shutil.rmtree(output_A)
    if os.path.exists(output_B):
        shutil.rmtree(output_B)
    
    os.makedirs(output_A, exist_ok=True)
    os.makedirs(output_B, exist_ok=True)
    
    for filename in tqdm(os.listdir(input_folder)):
        path = os.path.join(input_folder, filename)
        img = cv2.imread(path)
    
        h, w, _ = img.shape
        midpoint = w // 2
    
        left = img[:, :midpoint]
        right = img[:, midpoint:]
    
        cv2.imwrite(os.path.join(output_A, filename), left)
        cv2.imwrite(os.path.join(output_B, filename), right)

tf.keras.mixed_precision.set_global_policy('mixed_float16')

train_path = "/kaggle/input/datasets/balraj98/edges2shoes-dataset/train"
train_X = "/kaggle/working/train_X"
train_y = "/kaggle/working/train_y"

val_path = "/kaggle/input/datasets/balraj98/edges2shoes-dataset/val"
val_X = "/kaggle/working/val_X"
val_y = "/kaggle/working/val_y"

split_imgs(val_path, val_X, val_y)
split_imgs(train_path, train_X, train_y)

print(len(os.listdir('/kaggle/working/train_X')))
print(len(os.listdir('/kaggle/working/train_y')))

img_height, img_width = 256, 256
batch_size = 8

X = tf.keras.utils.image_dataset_from_directory(
  train_X,
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  label_mode=None,
  shuffle=False)

y = tf.keras.utils.image_dataset_from_directory(
  train_y,
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  label_mode=None,
  shuffle=False)

X_val = tf.keras.utils.image_dataset_from_directory(
  val_X,
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  label_mode=None,
  shuffle=False)

y_val = tf.keras.utils.image_dataset_from_directory(
  val_y,
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  label_mode=None,
  shuffle=False)

X_val = X_val.map(lambda x: x / 255.0).prefetch(tf.data.AUTOTUNE)
y_val = y_val.map(lambda x: x / 255.0).prefetch(tf.data.AUTOTUNE)

X = X.map(lambda x: x / 255.0).prefetch(tf.data.AUTOTUNE)
y = y.map(lambda x: x / 255.0).prefetch(tf.data.AUTOTUNE)

inputs = tf.keras.layers.Input(shape=(256,256,3))

c1 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
c1 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(c1)

p1 = tf.keras.layers.MaxPooling2D()(c1)

b = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(p1)
b = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(b)

u1 = tf.keras.layers.UpSampling2D()(b)

u1 = tf.keras.layers.Concatenate()([u1, c1])

c2 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(u1)
c2 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(c2)

outputs = tf.keras.layers.Conv2D(3,1, activation='sigmoid', padding='same', dtype='float32')(c2)

model = tf.keras.Model(inputs, outputs)

model.compile(optimizer='adam', loss='mae')

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

dataset = tf.data.Dataset.zip((X, y))
val_dataset = tf.data.Dataset.zip((X_val, y_val))

model.fit(dataset, epochs=20, callbacks=[early_stopping], validation_data=val_dataset)
model.save('/kaggle/working/edges-to-shoes.keras')

In [ ]:
# Testing if dataset exists
import os
os.listdir("/kaggle/input/datasets/balraj98/edges2shoes-dataset")